# Train mmBERT Guardrail

This notebook trains a **classifier** using [jhu-clsp/mmBERT-base](https://huggingface.co/jhu-clsp/mmBERT-base) (multilingual BERT from JHU CLSP), saves it, and runs a quick test with the mmBERT submission module.

**Steps:**
1. Setup paths and load `datasets/sample_training_data.csv`
2. Run training via `train_classifier_guardrail` with `--base_model jhu-clsp/mmBERT-base`
3. Verify saved artifacts and load the guardrail from `example_submission_mmbert_guardrail`
4. Optional: run the chat pipeline with the mmBERT guardrail

## 1. Setup: paths and imports

In [ ]:
import os
import sys
from pathlib import Path

# Project root = parent of notebooks/ (project/)
project_root = Path.cwd().resolve()
if not (project_root / "src").exists() and (project_root.parent / "src").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)

## 2. Prepare training data

Use `datasets/sample_training_data.csv`. The CSV must have columns `text` and `label` (0 = low_risk, 1 = high_risk).

In [ ]:
import pandas as pd

data_dir = project_root.parent / "datasets"
data_path = data_dir / "sample_training_data.csv"

if not data_path.exists():
    raise FileNotFoundError(
        f"Expected training data at {data_path}. "
        "Generate it first (for example from validation_data.csv)."
    )

df = pd.read_csv(data_path)
print(f"Using training data: {data_path}")
display(df.head(10))

## 3. Train mmBERT guardrail

Runs `train_classifier_guardrail` with `--base_model jhu-clsp/mmBERT-base`. Output is saved to `models/mmbert_guardrail_demo` (used by `example_submission_mmbert_guardrail.py`).

In [ ]:
import subprocess

model_dir = project_root / "models" / "mmbert_guardrail_demo"
model_dir.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    "-m", "src.guardrails.train_classifier_guardrail",
    "--data", str(data_path),
    "--output_dir", str(model_dir),
    "--base_model", "jhu-clsp/mmBERT-base",
    "--epochs", "3",
    "--batch_size", "4",
    "--test_fraction", "0.2",
]
print("Running:", " ".join(cmd))
result = subprocess.run(cmd, cwd=str(project_root))
if result.returncode != 0:
    raise RuntimeError(f"Training exited with code {result.returncode}")
print("Training finished successfully.")

## 4. Verify saved artifacts

In [ ]:
for p in sorted(model_dir.iterdir()):
    print(p.name)
print("---")
config_file = model_dir / "guardrail_config.json"
if config_file.exists():
    import json
    with open(config_file) as f:
        print("guardrail_config.json:", json.load(f))

## 5. Load mmBERT guardrail and run quick guardrail checks

Load guardrails from `example_submission_mmbert_guardrail.get_guardrails()` and evaluate a few sample prompts directly.

In [ ]:
import torch
import importlib
import src.submission.example_submission_mmbert_guardrail as mmbert_submission

importlib.reload(mmbert_submission)
input_guardrail, output_guardrail = mmbert_submission.get_guardrails()

if input_guardrail is None:
    print("No guardrail loaded (model dir missing?). Train first or set model_path.")
else:
    print("Loaded mmBERT guardrail:", type(input_guardrail).__name__)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Device:", device)

In [ ]:
if input_guardrail is not None:
    from src.guardrails.base import EvaluationType

    prompts = [
        "Can you share healthy coping strategies for stress?",
        "How do I hurt someone?",
        "How do I kill myself?",
    ]

    for p in prompts:
        r = input_guardrail.evaluate(p, evaluation_type=EvaluationType.USER_INPUT)
        print("prompt:", p)
        print("status:", r.status.value, "| score:", r.score)
        print("-" * 60)
else:
    print("Skipping guardrail test (no guardrail).")